# Phase 5: retrain with fog, and with fog + copy-paste

1. Choose **Runtime > Change runtime type > T4 GPU**.
2. `constellation_resplit.zip` should still be in the Drive folder `constellation` from Phase 2.
3. Choose **Runtime > Run all** and allow Drive access.

It builds two training sets (about 15 minutes), then trains two models for 50 epochs each (about 40 minutes each). If the session disconnects, run all again: finished runs are skipped and an interrupted one continues from its last epoch.
When it finishes, download `constellation/phase5_results.zip` from Drive.


In [ ]:
!pip -q install ultralytics==8.4.210 pyyaml
import torch
assert torch.cuda.is_available(), 'No GPU: choose Runtime > Change runtime type > T4 GPU'
print(torch.cuda.get_device_name(0))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import hashlib, pathlib, zipfile, yaml
DRIVE = pathlib.Path('/content/drive/MyDrive/constellation')
CLEAN = pathlib.Path('/content/data')
ZIP = DRIVE / 'constellation_resplit.zip'
assert ZIP.exists(), f'Upload the dataset zip to {ZIP}'
EXPECTED_SHA256 = '870fa2ffe1d2042a1173d40c4c75beef799b48922644a53642b3e3b18bab6424'
actual = hashlib.sha256(ZIP.read_bytes()).hexdigest()
assert actual == EXPECTED_SHA256, f'Wrong or damaged zip: {actual}'
if not (CLEAN / 'data.yaml').exists():
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(CLEAN)
print('Clean dataset ready')


In [ ]:
import os
os.makedirs('/content/src/constellation', exist_ok=True)
%cd /content


In [ ]:
%%writefile /content/src/constellation/fog_sim.py
"""Simulate fog and low light on night-sky images.

The model works in linear light (sRGB decoded) and follows the atmospheric
scattering model used for haze, I = J*t + A*(1 - t), with three additions
that matter for a night sky:

  1. Patchy fog: the optical depth varies smoothly across the frame, so the
     transmission t(x) = exp(-beta * (1 + patch * F(x))) is not constant.
  2. Scattered starlight: light removed from the direct beam reappears as a
     wide glow around bright sources (Moon, bright stars), and the direct
     beam itself is slightly blurred.
  3. Low light: the scene is scaled by an exposure factor, then sensor noise
     is added (Poisson shot noise for a given full-well count, plus Gaussian
     read noise), and the result is quantised to 8 bits.

Each image gets its own random generator, seeded from the file name, the
severity and --seed, so every run produces identical images.

Labels are copied unchanged: fog does not move the constellations.

Example:
  python -m src.constellation.fog_sim --src data/raw/constellations/resplit \
      --split test --out data/raw/constellations/fog
"""
import argparse
import hashlib
import json
import shutil
import sys
from dataclasses import asdict, dataclass
from pathlib import Path

import cv2
import numpy as np
import yaml

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}


@dataclass(frozen=True)
class FogParams:
    beta: float        # mean optical depth; t = exp(-beta) on average
    patch: float       # relative spread of optical depth across the frame (0 = uniform)
    airlight: float    # fog brightness in linear light (0..1)
    blur: float        # Gaussian sigma (px) of the direct beam
    glow: float        # fraction of scattered light that reappears as a halo
    glow_sigma: float  # Gaussian sigma (px) of the halo
    exposure: float    # scene scale for low light (1 = unchanged)
    full_well: float   # photons at linear value 1 after exposure; lower = more shot noise
    read_noise: float  # Gaussian read noise, std in linear light


SEVERITIES = {
    "light":  FogParams(beta=0.4, patch=0.3, airlight=0.010, blur=0.6, glow=0.15,
                        glow_sigma=12, exposure=0.6, full_well=4000, read_noise=0.0010),
    "medium": FogParams(beta=0.9, patch=0.4, airlight=0.018, blur=1.0, glow=0.20,
                        glow_sigma=16, exposure=0.4, full_well=2000, read_noise=0.0015),
    "heavy":  FogParams(beta=1.6, patch=0.5, airlight=0.028, blur=1.5, glow=0.25,
                        glow_sigma=20, exposure=0.25, full_well=1000, read_noise=0.0020),
}

# One component of "light" at a time, to see which part hurts detection.
# "identity" passes images through the same decode/encode path with no
# degradation, as a control for the pipeline itself.
_OFF = dict(beta=0.0, patch=0.0, airlight=0.0, blur=0.0, glow=0.0, glow_sigma=1.0,
            exposure=1.0, full_well=1e9, read_noise=0.0)
_L = SEVERITIES["light"]
ABLATIONS = {
    "identity": FogParams(**_OFF),
    "light_fog_only": FogParams(**{**_OFF, "beta": _L.beta, "patch": _L.patch, "airlight": _L.airlight,
                                   "blur": _L.blur, "glow": _L.glow, "glow_sigma": _L.glow_sigma}),
    "light_dark_only": FogParams(**{**_OFF, "exposure": _L.exposure}),
    "light_noise_only": FogParams(**{**_OFF, "full_well": _L.full_well, "read_noise": _L.read_noise}),
}
ALL_PARAMS = {**SEVERITIES, **ABLATIONS}

# Light-pollution tint of the airlight, BGR: slightly warm grey.
AIRLIGHT_TINT_BGR = np.array([0.85, 0.95, 1.0], dtype=np.float32)


def srgb_to_linear(x: np.ndarray) -> np.ndarray:
    return np.where(x <= 0.04045, x / 12.92, ((x + 0.055) / 1.055) ** 2.4)


def linear_to_srgb(x: np.ndarray) -> np.ndarray:
    x = np.clip(x, 0.0, 1.0)
    return np.where(x <= 0.0031308, 12.92 * x, 1.055 * np.power(x, 1 / 2.4) - 0.055)


def image_rng(name: str, severity: str, seed: int) -> np.random.Generator:
    digest = hashlib.sha256(f"{seed}:{severity}:{name}".encode()).digest()
    return np.random.default_rng(int.from_bytes(digest[:8], "little"))


def smooth_field(h: int, w: int, rng: np.random.Generator, cell: int = 80) -> np.ndarray:
    """Smooth random field in [-1, 1] with features about `cell` pixels wide."""
    gh, gw = max(2, h // cell + 2), max(2, w // cell + 2)
    coarse = rng.standard_normal((gh, gw)).astype(np.float32)
    field = cv2.resize(coarse, (w, h), interpolation=cv2.INTER_CUBIC)
    field = cv2.GaussianBlur(field, (0, 0), cell / 3)
    return field / (np.abs(field).max() + 1e-8)


def blur(x: np.ndarray, sigma: float) -> np.ndarray:
    return cv2.GaussianBlur(x, (0, 0), sigma) if sigma > 0 else x


def apply_fog(img_bgr: np.ndarray, p: FogParams, rng: np.random.Generator) -> np.ndarray:
    """uint8 BGR in, uint8 BGR out."""
    if img_bgr.dtype != np.uint8 or img_bgr.ndim != 3:
        raise ValueError("expected a uint8 BGR image")
    h, w = img_bgr.shape[:2]
    lin = srgb_to_linear(img_bgr.astype(np.float32) / 255.0).astype(np.float32)

    depth = p.beta * (1.0 + p.patch * smooth_field(h, w, rng))
    t = np.exp(-np.maximum(depth, 0.0))[..., None]

    direct = t * blur(lin, p.blur)
    halo = p.glow * (1.0 - t) * blur(lin, p.glow_sigma)
    air = p.airlight * AIRLIGHT_TINT_BGR * (1.0 - t)
    scene = p.exposure * (direct + halo + air)

    photons = rng.poisson(np.clip(scene, 0, None) * p.full_well).astype(np.float32)
    noisy = photons / p.full_well + rng.normal(0.0, p.read_noise, scene.shape).astype(np.float32)
    return np.round(linear_to_srgb(noisy) * 255.0).astype(np.uint8)


def fog_split(src: Path, split: str, out: Path, severity: str, seed: int, quality: int) -> dict:
    p = ALL_PARAMS[severity]
    img_in, lab_in = src / split / "images", src / split / "labels"
    img_out, lab_out = out / severity / split / "images", out / severity / split / "labels"
    img_out.mkdir(parents=True, exist_ok=True)
    lab_out.mkdir(parents=True, exist_ok=True)
    images = sorted(f for f in img_in.iterdir() if f.suffix.lower() in IMG_EXTS)
    for f in images:
        img = cv2.imread(str(f), cv2.IMREAD_COLOR)
        if img is None:
            raise RuntimeError(f"cannot read {f}")
        fogged = apply_fog(img, p, image_rng(f.name, severity, seed))
        cv2.imwrite(str(img_out / (f.stem + ".jpg")), fogged, [cv2.IMWRITE_JPEG_QUALITY, quality])
        lab = lab_in / (f.stem + ".txt")
        if lab.exists():
            shutil.copyfile(lab, lab_out / lab.name)
    return {"images": len(images)}


def write_data_yaml(src: Path, out_dir: Path, splits: list[str]) -> None:
    cfg = yaml.safe_load((src / "data.yaml").read_text())
    new = {"path": str(out_dir.resolve()), "names": cfg["names"], "nc": cfg.get("nc", len(cfg["names"]))}
    for key, split in (("train", "train"), ("val", "valid"), ("test", "test")):
        # Splits that were not fogged point at the clean images, so the yaml always loads.
        new[key] = (f"{split}/images" if split in splits
                    else str((src / split / "images").resolve()))
    (out_dir / "data.yaml").write_text(yaml.safe_dump(new, sort_keys=False))


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__,
                                 formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--src", type=Path, default=Path("data/raw/constellations/resplit"))
    ap.add_argument("--out", type=Path, default=Path("data/raw/constellations/fog"))
    ap.add_argument("--split", nargs="+", default=["test"], choices=("train", "valid", "test"))
    ap.add_argument("--severity", nargs="+", default=list(SEVERITIES), choices=list(ALL_PARAMS),
                    help="fog levels, or ablations that apply one part of 'light' at a time")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--quality", type=int, default=95, help="JPEG quality of the output")
    args = ap.parse_args()

    report = {"seed": args.seed, "jpeg_quality": args.quality, "src": str(args.src), "severities": {}}
    for sev in args.severity:
        counts = {s: fog_split(args.src, s, args.out, sev, args.seed, args.quality) for s in args.split}
        write_data_yaml(args.src, args.out / sev, args.split)
        report["severities"][sev] = {"params": asdict(ALL_PARAMS[sev]), "splits": counts}
        print(sev, counts)
    args.out.mkdir(parents=True, exist_ok=True)
    (args.out / f"fog_report_{'_'.join(args.severity)}.json").write_text(json.dumps(report, indent=2))
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile /content/src/constellation/make_train_sets.py
"""Build the Phase 5 training sets from the clean re-split.

  fogmix     every train and valid image gets one condition, drawn uniformly
             from clean / light / medium / heavy, and is degraded with
             fog_sim.py. Same images and labels as the baseline, only degraded.
  fogmix_cp  like fogmix, but before the fog, half of the train images get
             1-2 constellations copied in from other train images
             (copy-paste). Valid is the same as fogmix (no pasting).

Copy-paste for a night sky: the pasted crop's own background level (its
per-channel median) is subtracted and only the light above it is added to
the target image, so stars are added without a visible rectangle. A crop is
placed only where it does not overlap any box already in the image.

The test split is not touched: both data.yaml files point at the clean test
images, and the fogged test sets from Phase 3 are used for scoring.

Example:
  python -m src.constellation.make_train_sets --src data/raw/constellations/resplit \
      --out data/raw/constellations/phase5
"""
import argparse
import csv
import hashlib
import shutil
import sys
from pathlib import Path

import cv2
import numpy as np
import yaml

from src.constellation.fog_sim import SEVERITIES, apply_fog, image_rng

CONDITIONS = ["clean", *SEVERITIES]
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}


def rng_for(name: str, tag: str, seed: int) -> np.random.Generator:
    d = hashlib.sha256(f"{seed}:{tag}:{name}".encode()).digest()
    return np.random.default_rng(int.from_bytes(d[:8], "little"))


def read_labels(path: Path) -> list[list[float]]:
    if not path.exists():
        return []
    return [[float(v) for v in line.split()[:5]] for line in path.read_text().splitlines() if line.strip()]


def write_labels(path: Path, rows: list[list[float]]) -> None:
    path.write_text("".join(f"{int(r[0])} {r[1]:.6f} {r[2]:.6f} {r[3]:.6f} {r[4]:.6f}\n" for r in rows))


def to_px(r, w, h):
    _, cx, cy, bw, bh = r
    return int((cx - bw / 2) * w), int((cy - bh / 2) * h), int(np.ceil((cx + bw / 2) * w)), int(np.ceil((cy + bh / 2) * h))


def crop_bank(images: list[Path], labels_dir: Path, max_frac: float) -> list[tuple]:
    """(source stem, class, crop) for every box at most max_frac of the image in each direction."""
    bank = []
    for f in images:
        img = cv2.imread(str(f))
        h, w = img.shape[:2]
        for r in read_labels(labels_dir / f"{f.stem}.txt"):
            if r[3] <= max_frac and r[4] <= max_frac:
                x0, y0, x1, y1 = to_px(r, w, h)
                x0, y0, x1, y1 = max(0, x0), max(0, y0), min(w, x1), min(h, y1)
                if x1 - x0 >= 8 and y1 - y0 >= 8:
                    bank.append((f.stem, int(r[0]), img[y0:y1, x0:x1].copy()))
    return bank


def overlaps(a, b) -> bool:
    return not (a[2] <= b[0] or b[2] <= a[0] or a[3] <= b[1] or b[3] <= a[1])


def paste(img, rows, bank, rng, stem, n_max, tries=30):
    """Add 1..n_max crops from other images where they overlap no existing box."""
    h, w = img.shape[:2]
    out = img.astype(np.float32)
    boxes = [to_px(r, w, h) for r in rows]
    pasted = []
    for _ in range(int(rng.integers(1, n_max + 1))):
        src_stem, cls, crop = bank[int(rng.integers(len(bank)))]
        if src_stem == stem:
            continue
        ch, cw = crop.shape[:2]
        for _ in range(tries):
            x0, y0 = int(rng.integers(0, w - cw + 1)), int(rng.integers(0, h - ch + 1))
            box = (x0, y0, x0 + cw, y0 + ch)
            if not any(overlaps(box, b) for b in boxes):
                c = crop.astype(np.float32)
                light = np.maximum(c - np.median(c.reshape(-1, 3), axis=0), 0)
                out[y0:y0 + ch, x0:x0 + cw] += light
                boxes.append(box)
                rows.append([cls, (x0 + cw / 2) / w, (y0 + ch / 2) / h, cw / w, ch / h])
                pasted.append(f"{cls}@{src_stem[:24]}")
                break
    return np.clip(out + 0.5, 0, 255).astype(np.uint8), rows, pasted


def build(src: Path, out: Path, seed: int, cp_prob: float, cp_max: int, max_frac: float, quality: int):
    names = yaml.safe_load((src / "data.yaml").read_text())["names"]
    manifest = []
    train_imgs = sorted(p for p in (src / "train/images").iterdir() if p.suffix.lower() in IMG_EXTS)
    bank = crop_bank(train_imgs, src / "train/labels", max_frac)
    print(f"copy-paste bank: {len(bank)} crops")
    jobs = [("fogmix", "train", False), ("fogmix", "valid", False), ("fogmix_cp", "train", True)]
    for set_name, split, do_cp in jobs:
        img_out, lab_out = out / set_name / split / "images", out / set_name / split / "labels"
        img_out.mkdir(parents=True, exist_ok=True)
        lab_out.mkdir(parents=True, exist_ok=True)
        images = sorted(p for p in (src / split / "images").iterdir() if p.suffix.lower() in IMG_EXTS)
        for i, f in enumerate(images):
            img = cv2.imread(str(f))
            rows = read_labels(src / split / "labels" / f"{f.stem}.txt")
            # The condition depends only on the image, so fogmix and fogmix_cp
            # give each train image the same fog level.
            cond = CONDITIONS[int(rng_for(f.name, "condition", seed).integers(len(CONDITIONS)))]
            pasted = []
            if do_cp:
                r = rng_for(f.name, "paste", seed)
                if r.random() < cp_prob:
                    img, rows, pasted = paste(img, rows, bank, r, f.stem, cp_max)
            if cond != "clean":
                img = apply_fog(img, SEVERITIES[cond], image_rng(f.name, cond, seed))
            cv2.imwrite(str(img_out / f"{f.stem}.jpg"), img, [cv2.IMWRITE_JPEG_QUALITY, quality])
            write_labels(lab_out / f"{f.stem}.txt", rows)
            manifest.append([set_name, split, f.name, cond, ";".join(pasted)])
            if (i + 1) % 500 == 0:
                print(f"{set_name}/{split}: {i + 1}/{len(images)}", flush=True)
    # fogmix_cp validates on the same fogged valid images as fogmix.
    shutil.copytree(out / "fogmix" / "valid", out / "fogmix_cp" / "valid", dirs_exist_ok=True)
    for set_name in ("fogmix", "fogmix_cp"):
        cfg = {"path": str((out / set_name).resolve()), "train": "train/images", "val": "valid/images",
               "test": str((src / "test/images").resolve()), "names": names, "nc": len(names)}
        (out / set_name / "data.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False))
    with open(out / "manifest.csv", "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(["set", "split", "image", "condition", "pasted"])
        w.writerows(manifest)


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__,
                                 formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--src", type=Path, default=Path("data/raw/constellations/resplit"))
    ap.add_argument("--out", type=Path, default=Path("data/raw/constellations/phase5"))
    ap.add_argument("--seed", type=int, default=1, help="differs from the test fog seed (0)")
    ap.add_argument("--cp-prob", type=float, default=0.5, help="share of train images that get pasted crops")
    ap.add_argument("--cp-max", type=int, default=2, help="most crops pasted into one image")
    ap.add_argument("--max-frac", type=float, default=0.5, help="largest crop, as a fraction of the image side")
    ap.add_argument("--quality", type=int, default=95)
    args = ap.parse_args()
    build(args.src, args.out, args.seed, args.cp_prob, args.cp_max, args.max_frac, args.quality)
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile /content/src/constellation/train_yolo.py
"""Fine-tune a YOLO detector on the constellation dataset.

Every experiment (baseline, fog, fog + copy-paste) uses this script, so the
only thing that differs between them is --data and --name. Early stopping is
off by default so that each run gets the same number of epochs.

Example (Colab):
  python train_yolo.py --data /content/data/data.yaml --name baseline \
      --project /content/drive/MyDrive/constellation/runs
Continue an interrupted run by repeating the command with --resume.
"""
import argparse
import hashlib
import json
import sys
import time
from pathlib import Path

import yaml


def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__,
                                 formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--data", type=Path, default=Path("data/raw/constellations/resplit/data.yaml"))
    ap.add_argument("--model", default="yolov8n.pt")
    ap.add_argument("--name", default="baseline")
    ap.add_argument("--project", type=Path, default=Path("runs/constellation"))
    ap.add_argument("--epochs", type=int, default=100)
    ap.add_argument("--imgsz", type=int, default=640)
    ap.add_argument("--batch", type=int, default=16)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--patience", type=int, default=0, help="0 turns early stopping off")
    ap.add_argument("--device", default="0", help="GPU index, or cpu")
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--resume", action="store_true")
    args = ap.parse_args()

    import torch
    import ultralytics
    from ultralytics import YOLO

    project = args.project.resolve()
    run_dir = project / args.name
    start = time.time()
    if args.resume:
        last = run_dir / "weights" / "last.pt"
        if not last.exists():
            print(f"ERROR: nothing to resume at {last}", file=sys.stderr)
            return 1
        YOLO(str(last)).train(resume=True, device=args.device, workers=args.workers)
    else:
        if run_dir.exists():
            print(f"ERROR: {run_dir} already exists. Use --resume or a new --name.", file=sys.stderr)
            return 1
        YOLO(args.model).train(
            data=str(args.data.resolve()), epochs=args.epochs, imgsz=args.imgsz,
            batch=args.batch, seed=args.seed, deterministic=True, patience=args.patience,
            device=args.device, workers=args.workers, project=str(project), name=args.name)

    best = run_dir / "weights" / "best.pt"
    if not best.exists():
        print(f"ERROR: training finished but {best} is missing", file=sys.stderr)
        return 1
    # Read the settings the run really used, which on --resume are the original
    # run's rather than this command line's.
    used = yaml.safe_load((run_dir / "args.yaml").read_text())
    info = {
        "name": args.name, "model": used["model"], "data": used["data"],
        "data_yaml_sha256": sha256(Path(used["data"])), "epochs": used["epochs"],
        "imgsz": used["imgsz"], "batch": used["batch"], "seed": used["seed"],
        "patience": used["patience"],
        "resumed": args.resume, "seconds_this_session": round(time.time() - start, 1),
        "best_sha256": sha256(best), "ultralytics": ultralytics.__version__,
        "torch": torch.__version__,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    }
    (run_dir / "train_info.json").write_text(json.dumps(info, indent=2))
    print(json.dumps(info, indent=2))
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
SETS = pathlib.Path('/content/phase5')
if not (SETS / 'fogmix_cp' / 'data.yaml').exists():
    !python -m src.constellation.make_train_sets --src {CLEAN} --out {SETS}
assert (SETS / 'fogmix_cp' / 'data.yaml').exists(), 'Building the training sets failed; read the log above.'


In [ ]:
RUNS = DRIVE / 'runs'
for name in ['fogmix', 'fogmix_cp']:
    run = RUNS / name
    if (run / 'train_info.json').exists():
        print(name, 'already finished, skipping.')
        continue
    resume = '--resume' if (run / 'weights' / 'last.pt').exists() else ''
    data = SETS / name / 'data.yaml'
    !python src/constellation/train_yolo.py --data {data} --name {name} --project {RUNS} --epochs 50 --imgsz 640 --batch 16 --seed 0 --device 0 --workers 2 {resume}
    assert (run / 'train_info.json').exists(), f'{name} did not finish; read the log above.'


In [ ]:
with zipfile.ZipFile(DRIVE / 'phase5_results.zip', 'w') as z:
    z.write(SETS / 'manifest.csv', 'manifest.csv')
    for name in ['fogmix', 'fogmix_cp']:
        for f in ('results.csv', 'args.yaml', 'train_info.json', 'weights/best.pt'):
            z.write(RUNS / name / f, f'{name}/{f}')
print('Done. Download', DRIVE / 'phase5_results.zip')
